# DefectVision: one-class industrial defect detection (F1, defective = positive)

**Idea:** train only on GOOD images. Learn what normal patches look like using frozen
general-purpose ImageNet / DINOv2 features (allowed by the rules), store them in a memory bank
(PatchCore-style), and flag test images that contain patches far from anything normal.

**Pipeline**
1. Load + normalise all images (any size / format / grayscale / noise) -> RGB, fixed size.
2. Per category: build augmented memory banks (rotations, noise, JPEG, grayscale) with greedy coreset.
3. K-fold on the good images -> honest "good" score distribution + synthetic-defect pseudo-validation.
4. Ensemble of 3 models (WideResNet-50, WideResNet-101, DINOv2) on normalised scores.
5. Per-category thresholds (3 methods + blend), writes several submission files.

**Kaggle setup:** Settings -> Accelerator = GPU (T4 / P100), Internet = ON (to download the
ImageNet / DINOv2 weights). Run all cells. Results are cached in /kaggle/working/scores so a
re-run after changing thresholds only takes seconds.

In [ ]:
import os, io, glob, gc, json, math, random, time, warnings
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFilter, ImageEnhance
import torch
import torch.nn.functional as F
import torchvision
from sklearn.metrics import roc_auc_score, precision_recall_curve, f1_score
from sklearn.model_selection import KFold
from sklearn.mixture import GaussianMixture
import matplotlib
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# ----------------------------- CONFIG ---------------------------------------------------------
SEED = 42
QUICK = bool(int(os.environ.get("QUICK", "0")))          # 1 = fast first submission (1 model, 3 folds)
PRETRAINED = bool(int(os.environ.get("PRETRAINED", "1")))  # keep 1 on Kaggle (0 only for smoke tests)
MODELS = os.environ.get("MODELS", "wrn50,wrn101,dinov2").split(",")
if QUICK:
    MODELS = ["wrn50"]
N_FOLDS = 3 if QUICK else 4
BASE = int(os.environ.get("BASE", "320"))   # all images are resized to BASE x BASE first
DINO_RES = int(os.environ.get("DINO_RES", "448"))
FEAT_DIM = 1024
CORESET_RATIO = 0.10       # fraction of patches kept in memory bank
MAX_BANK = 250_000         # hard cap of bank size (patches)
CHUNK_VIEWS = 64           # views per coreset chunk
BATCH = 16
BANK_CHUNK = 40_000
SYN_PER_GOOD = 2           # synthetic defects generated per good image (pseudo-validation)
PCT = 97                   # percentile of held-out good scores used for threshold method 2
TARGET_DEF_FRAC = 0.65     # test is defect-heavy; pseudo-validation is weighted to this defect share
CATS = ["bottle", "cable", "screw", "hazelnut"]

# per-category augmentation of the NORMAL memory bank.
# screw / hazelnut / bottle appear at arbitrary rotations; cable has a fixed wire layout -> no rotation.
# screw is never flipped (mirrored thread would look defective).
CAT_CFG = {
    "bottle":   {"rots": [0, 90, 180, 270], "flip": False},
    "cable":    {"rots": [0], "flip": False},
    "screw":    {"rots": [0, 45, 90, 135, 180, 225, 270, 315], "flip": False},
    "hazelnut": {"rots": [0, 90, 180, 270], "flip": True},
}

DEV = "cuda" if torch.cuda.is_available() else "cpu"
OUT = os.environ.get("OUT_DIR", "/kaggle/working")
os.makedirs(f"{OUT}/scores", exist_ok=True)
os.makedirs(f"{OUT}/figs", exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("device:", DEV, "| models:", MODELS, "| folds:", N_FOLDS)

## 1. Load data (robust to mixed size / format / grayscale / noise)

In [ ]:
def find_root():
    r = os.environ.get("DATA_ROOT")
    if r:
        return r
    hits = glob.glob("/kaggle/input/**/train.csv", recursive=True)
    if not hits:
        raise FileNotFoundError("train.csv not found - add the competition data to the notebook")
    return os.path.dirname(hits[0])

ROOT = find_root()
train = pd.read_csv(f"{ROOT}/train.csv")
test = pd.read_csv(f"{ROOT}/test.csv")
sample_path = f"{ROOT}/sample_submission.csv"
print("ROOT:", ROOT)
print(train.groupby("category").size().to_dict(), "| test:", test.groupby("category").size().to_dict())
assert (train["label"] == "good").all()

_index = None
def resolve(p):
    global _index
    for c in (f"{ROOT}/clips/{p}", f"{ROOT}/{p}", p):
        if os.path.exists(c):
            return c
    if _index is None:
        _index = {os.path.basename(f): f for f in glob.glob(f"{ROOT}/**/*.*", recursive=True)}
    return _index[os.path.basename(p)]

def noise_sigma(gray):
    """Immerkaer fast noise-level estimate on the ORIGINAL resolution gray image."""
    g = gray.astype(np.float32)
    H, W = g.shape
    if H < 3 or W < 3:
        return 0.0
    c = (g[:-2, :-2] - 2 * g[:-2, 1:-1] + g[:-2, 2:] - 2 * g[1:-1, :-2] + 4 * g[1:-1, 1:-1]
         - 2 * g[1:-1, 2:] + g[2:, :-2] - 2 * g[2:, 1:-1] + g[2:, 2:])
    return float(np.sqrt(np.pi / 2) / (6.0 * (W - 2) * (H - 2)) * np.abs(c).sum())

def load_one(path):
    im = Image.open(resolve(path)); im.load()
    rgb = im.convert("RGB")                       # grayscale / RGBA / palette -> RGB
    sig = noise_sigma(np.asarray(rgb.convert("L")))
    arr = np.asarray(rgb.resize((BASE, BASE), Image.BICUBIC))
    return arr, sig

def load_df(df):
    arrs, sigs = [], []
    for p in df["path"]:
        a, s = load_one(p)
        arrs.append(a); sigs.append(s)
    return arrs, np.array(sigs)

DATA = {}
for cat in CATS:
    tr = train[train.category == cat].reset_index(drop=True)
    te = test[test.category == cat].reset_index(drop=True)
    tr_a, tr_s = load_df(tr)
    te_a, te_s = load_df(te)
    allsig = np.concatenate([tr_s, te_s])
    med = np.median(allsig); mad = 1.4826 * np.median(np.abs(allsig - med))
    thr = med + 6 * max(mad, 0.25)
    DATA[cat] = dict(tr=tr, te=te, tr_a=tr_a, te_a=te_a, tr_noisy=tr_s > thr, te_noisy=te_s > thr)
    print(f"{cat:9s} train {len(tr):4d} test {len(te):4d} | noise thr {thr:5.2f} | noisy frac "
          f"train {np.mean(tr_s > thr):.2f} test {np.mean(te_s > thr):.2f}")

# quick look at the data
fig, axes = plt.subplots(len(CATS), 6, figsize=(14, 2.4 * len(CATS)))
for r, cat in enumerate(CATS):
    for c in range(6):
        axes[r, c].imshow(DATA[cat]["tr_a"][c]); axes[r, c].axis("off")
    axes[r, 0].set_title(cat, fontsize=9, loc="left")
plt.tight_layout(); plt.savefig(f"{OUT}/figs/eda_train.png", dpi=70); plt.show()

## 2. Augmentation, degradations, synthetic defects (pseudo-validation)

In [ ]:
def rotate_arr(a, ang):
    if ang == 0:
        return a
    if ang % 90 == 0:
        return np.ascontiguousarray(np.rot90(a, k=ang // 90))
    border = np.concatenate([a[0], a[-1], a[:, 0], a[:, -1]], 0)
    fill = tuple(int(v) for v in np.median(border, 0))
    return np.asarray(Image.fromarray(a).rotate(ang, resample=Image.BILINEAR, fillcolor=fill))

def d_noise(a, rng):
    s = rng.uniform(4, 20)
    return np.clip(a.astype(np.float32) + rng.normal(0, s, a.shape), 0, 255).astype(np.uint8)

def d_jpeg(a, rng):
    buf = io.BytesIO()
    Image.fromarray(a).save(buf, format="JPEG", quality=int(rng.integers(30, 90)))
    return np.asarray(Image.open(buf).convert("RGB"))

def d_gray(a, rng):
    g = np.asarray(Image.fromarray(a).convert("L"))
    return np.stack([g] * 3, -1)

def make_views(a, cfg, rng):
    """Augmented views of a NORMAL image for the memory bank."""
    views = []
    for ang in cfg["rots"]:
        v = rotate_arr(a, ang)
        views.append(v)
        if cfg["flip"]:
            views.append(np.ascontiguousarray(v[:, ::-1]))
    for fn in (d_noise, d_jpeg, d_gray):            # sensor-style variation seen in the test set
        ang = int(rng.choice(cfg["rots"]))
        views.append(fn(rotate_arr(a, ang), rng))
    return views

def make_synthetic(arr, rng):
    """Cheap synthetic defect (scratch / blob / cut-paste / erase / dots / smudge) for pseudo-validation."""
    H, W = arr.shape[:2]
    img = Image.fromarray(arr.copy())
    kind = rng.choice(["cutpaste", "scratch", "blob", "erase", "dots", "smudge"])
    lo, hi = int(0.18 * W), int(0.82 * W)

    def cxy(w, h):
        return (int(rng.integers(lo, max(lo + 1, hi - w))), int(rng.integers(lo, max(lo + 1, hi - h))))

    def color(dark):
        y, x = int(rng.integers(0, H)), int(rng.integers(0, W))
        c = arr[y, x].astype(np.float32)
        f = rng.uniform(0.15, 0.6) if dark else rng.uniform(1.4, 2.0)
        return tuple(int(v) for v in np.clip(c * f + (0 if dark else 20), 0, 255))

    if kind == "cutpaste":
        pw, ph = int(rng.integers(12, 60)), int(rng.integers(12, 60))
        sx, sy = int(rng.integers(0, W - pw)), int(rng.integers(0, H - ph))
        patch = img.crop((sx, sy, sx + pw, sy + ph))
        patch = patch.rotate(float(rng.uniform(0, 360)), expand=True)
        patch = ImageEnhance.Brightness(patch).enhance(float(rng.uniform(0.6, 1.4)))
        img.paste(patch, cxy(*patch.size))
    elif kind == "scratch":
        d = ImageDraw.Draw(img)
        x, y = cxy(30, 30)
        col = color(bool(rng.integers(0, 2)))
        for _ in range(int(rng.integers(2, 5))):
            nx = int(np.clip(x + rng.integers(-35, 36), 0, W - 1)); ny = int(np.clip(y + rng.integers(-35, 36), 0, H - 1))
            d.line([(x, y), (nx, ny)], fill=col, width=int(rng.integers(1, 4)))
            x, y = nx, ny
    elif kind in ("blob", "erase"):
        rw, rh = int(rng.integers(5, 26)), int(rng.integers(5, 26))
        x, y = cxy(2 * rw, 2 * rh)
        mask = Image.new("L", (W, H), 0)
        md = ImageDraw.Draw(mask)
        if kind == "blob":
            md.ellipse([x, y, x + 2 * rw, y + 2 * rh], fill=255)
        else:
            md.rectangle([x, y, x + 2 * rw, y + 2 * rh], fill=255)
        mask = mask.filter(ImageFilter.GaussianBlur(1.5))
        layer = Image.new("RGB", (W, H), color(bool(rng.integers(0, 2))))
        img = Image.composite(layer, img, mask)
    elif kind == "dots":
        d = ImageDraw.Draw(img)
        col = color(True)
        for _ in range(int(rng.integers(3, 12))):
            x, y = cxy(6, 6); r = int(rng.integers(1, 4))
            d.ellipse([x, y, x + r, y + r], fill=col)
    else:  # smudge
        w, h = int(rng.integers(20, 60)), int(rng.integers(20, 60))
        x, y = cxy(w, h)
        patch = img.crop((x, y, x + w, y + h)).filter(ImageFilter.GaussianBlur(5))
        patch = ImageEnhance.Brightness(patch).enhance(float(rng.uniform(0.5, 0.9)))
        img.paste(patch, (x, y))
    return np.asarray(img)

SYN = {}
for cat in CATS:
    rng = np.random.default_rng(SEED + 7)
    SYN[cat] = [[make_synthetic(a, rng) for _ in range(SYN_PER_GOOD)] for a in DATA[cat]["tr_a"]]

fig, axes = plt.subplots(len(CATS), 6, figsize=(14, 2.4 * len(CATS)))
for r, cat in enumerate(CATS):
    for c in range(6):
        axes[r, c].imshow(SYN[cat][c][0]); axes[r, c].axis("off")
    axes[r, 0].set_title(cat + " (synthetic defects used for validation)", fontsize=9, loc="left")
plt.tight_layout(); plt.savefig(f"{OUT}/figs/synthetic_examples.png", dpi=70); plt.show()

## 3. Feature extractors (frozen, general-purpose pretrained backbones) + PatchCore memory bank

In [ ]:
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STDV = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

def prep(arrs, size=None):
    x = torch.from_numpy(np.stack(arrs)).permute(0, 3, 1, 2).float().div(255).to(DEV)
    if size and size != x.shape[-1]:
        x = F.interpolate(x, size=(size, size), mode="bicubic", align_corners=False).clamp(0, 1)
    return (x - MEAN.to(DEV)) / STDV.to(DEV)

class CNNFeat:
    """PatchCore features: WideResNet layer2+layer3, 3x3 local pooling, projected to FEAT_DIM."""
    def __init__(self, name):
        tv = torchvision.models
        if name == "wrn50":
            m = tv.wide_resnet50_2(weights=tv.Wide_ResNet50_2_Weights.IMAGENET1K_V1 if PRETRAINED else None)
        else:
            m = tv.wide_resnet101_2(weights=tv.Wide_ResNet101_2_Weights.IMAGENET1K_V1 if PRETRAINED else None)
        self.m = m.to(DEV).eval()
        for p in self.m.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def __call__(self, arrs):
        x = prep(arrs); m = self.m
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(DEV == "cuda")):
            x = m.maxpool(m.relu(m.bn1(m.conv1(x))))
            x = m.layer1(x); l2 = m.layer2(x); l3 = m.layer3(l2)
        l2 = F.avg_pool2d(l2.float(), 3, 1, 1); l3 = F.avg_pool2d(l3.float(), 3, 1, 1)
        l3 = F.interpolate(l3, size=l2.shape[-2:], mode="bilinear", align_corners=False)
        f = torch.cat([l2, l3], 1)
        B, C, h, w = f.shape
        f = f.permute(0, 2, 3, 1).reshape(B * h * w, 1, C)
        return F.adaptive_avg_pool1d(f, FEAT_DIM).reshape(B, h * w, FEAT_DIM)

class DinoFeat:
    """DINOv2-base patch tokens (last layer + layer 8, L2-normalised) at DINO_RES."""
    def __init__(self):
        from transformers import AutoModel
        self.m = AutoModel.from_pretrained("facebook/dinov2-base").to(DEV).eval()
        for p in self.m.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def __call__(self, arrs):
        x = prep(arrs, DINO_RES)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(DEV == "cuda")):
            hs = self.m(pixel_values=x, output_hidden_states=True).hidden_states
        a = F.normalize(hs[-1][:, 1:].float(), dim=-1)
        b = F.normalize(hs[8][:, 1:].float(), dim=-1)
        return torch.cat([a, b], -1)

def make_extractor(name):
    return DinoFeat() if name == "dinov2" else CNNFeat(name)

def patches(ext, arrs):
    outs = [ext(arrs[i:i + BATCH]) for i in range(0, len(arrs), BATCH)]
    f = torch.cat(outs)
    return f.reshape(-1, f.shape[-1])

@torch.no_grad()
def greedy_coreset(X, k, seed=0):
    """Greedy k-center coreset on a random 128-d projection (PatchCore subsampling)."""
    N = X.shape[0]
    if k >= N:
        return X
    g = torch.Generator(device=X.device); g.manual_seed(seed)
    P = torch.randn(X.shape[1], 128, generator=g, device=X.device) / math.sqrt(128)
    Z = X @ P
    idx = torch.zeros(k, dtype=torch.long, device=X.device)
    j = torch.randint(0, N, (1,), generator=g, device=X.device)[0]
    idx[0] = j
    mind = (Z - Z[j]).norm(dim=1)
    for i in range(1, k):
        j = torch.argmax(mind)
        idx[i] = j
        mind = torch.minimum(mind, (Z - Z[j]).norm(dim=1))
    return X[idx]

def build_bank(ext, arrs, cfg, seed):
    rng = np.random.default_rng(seed)
    nviews = len(cfg["rots"]) * (2 if cfg["flip"] else 1) + 3
    npatch = ext([arrs[0]]).shape[1]
    ratio = min(CORESET_RATIO, MAX_BANK / (len(arrs) * nviews * npatch))
    bank, buf = [], []

    def flush():
        X = patches(ext, buf)
        bank.append(greedy_coreset(X, max(1, int(ratio * X.shape[0])), seed))
        buf.clear()

    for a in arrs:
        buf.extend(make_views(a, cfg, rng))
        if len(buf) >= CHUNK_VIEWS:
            flush()
    if buf:
        flush()
    return torch.cat(bank)

@torch.no_grad()
def score_images(ext, bank, arrs):
    """Anomaly score = distance of the most unusual patch to the normal bank. Returns (n, 2): max, top-5 mean."""
    if len(arrs) == 0:
        return np.zeros((0, 2))
    bsq = (bank ** 2).sum(1)
    res = []
    for i in range(0, len(arrs), BATCH):
        f = ext(arrs[i:i + BATCH])
        for b in range(f.shape[0]):
            q = f[b]; qsq = (q ** 2).sum(1)
            mind = torch.full((q.shape[0],), float("inf"), device=q.device)
            for j in range(0, bank.shape[0], BANK_CHUNK):
                d = qsq[:, None] + bsq[None, j:j + BANK_CHUNK] - 2 * q @ bank[j:j + BANK_CHUNK].T
                mind = torch.minimum(mind, d.min(1).values)
            ps = mind.clamp_min(0).sqrt()
            res.append([ps.max().item(), torch.topk(ps, min(5, ps.numel())).values.mean().item()])
    return np.array(res)

## 4. Run every model x category (K-fold on good images + synthetic defects + test scoring)

In [ ]:
def run_model_cat(model_name, ext, cat):
    path = f"{OUT}/scores/{model_name}_{cat}.npz"
    if os.path.exists(path):
        print(f"  cached {model_name}/{cat}")
        return
    d = DATA[cat]; tr_a, te_a = d["tr_a"], d["te_a"]; cfg = CAT_CFG[cat]
    n = len(tr_a)
    oof = np.zeros((n, 2)); syn = np.zeros((n, SYN_PER_GOOD, 2))
    t0 = time.time()
    kf = KFold(N_FOLDS, shuffle=True, random_state=SEED)
    for f, (tri, hoi) in enumerate(kf.split(np.arange(n))):
        bank = build_bank(ext, [tr_a[i] for i in tri], cfg, SEED + f)
        oof[hoi] = score_images(ext, bank, [tr_a[i] for i in hoi])
        syn_list = [s for i in hoi for s in SYN[cat][i]]
        syn[hoi] = score_images(ext, bank, syn_list).reshape(len(hoi), SYN_PER_GOOD, 2)
        del bank; gc.collect(); torch.cuda.empty_cache()
    bank = build_bank(ext, tr_a, cfg, SEED + 99)                 # final bank from ALL good images
    test_s = score_images(ext, bank, te_a)
    del bank; gc.collect(); torch.cuda.empty_cache()
    np.savez(path, oof=oof, syn=syn, test=test_s)
    print(f"  done {model_name}/{cat} in {time.time() - t0:.0f}s")

ACTIVE = []
for mname in MODELS:
    try:
        ext = make_extractor(mname)
    except Exception as e:
        print(f"!! could not load {mname}: {e}  (skipping; is Internet ON?)")
        continue
    ACTIVE.append(mname)
    print("model:", mname)
    for cat in CATS:
        run_model_cat(mname, ext, cat)
    del ext; gc.collect(); torch.cuda.empty_cache()
print("active models:", ACTIVE)

## 5. Ensemble, pseudo-validation report, thresholds, submissions

In [ ]:
def norm_params(oof_s, noisy):
    """z-score using held-out GOOD scores, separately for noisy / clean images (noise inflates scores)."""
    P = {}
    for g in (False, True):
        m = noisy == g
        x = oof_s[m] if m.sum() >= 8 else oof_s
        P[g] = (x.mean(), x.std() + 1e-6)
    return P

def apply_norm(s, noisy, P):
    mu = np.where(noisy, P[True][0], P[False][0]); sd = np.where(noisy, P[True][1], P[False][1])
    return (s - mu) / sd

def best_f1(scores, y, w=None):
    p, r, t = precision_recall_curve(y, scores, sample_weight=w)
    f = 2 * p * r / np.clip(p + r, 1e-9, None)
    i = int(np.nanargmax(f[:-1]))
    return f[i], t[i]

def gmm_threshold(s):
    s = np.asarray(s).reshape(-1, 1)
    gm = GaussianMixture(2, random_state=0, n_init=5).fit(s)
    mu = gm.means_.ravel(); sd = np.sqrt(gm.covariances_.ravel()); lo, hi = np.argsort(mu)
    sep = (mu[hi] - mu[lo]) / np.sqrt(0.5 * (sd[lo] ** 2 + sd[hi] ** 2))
    grid = np.linspace(mu[lo], mu[hi], 2000)
    post = gm.predict_proba(grid.reshape(-1, 1))[:, hi]
    t = grid[int(np.argmax(post >= 0.5))]
    return t, sep, gm.weights_[hi]

SCORE_KEY = 0   # 0 = max patch score, 1 = mean of top-5 patch scores
rows, thr_info = [], {}
ens_test = {}
pred_modes = {m: {} for m in ["val", "pct", "gmm", "blend"]}
suspects = {}

for cat in CATS:
    d = DATA[cat]; n = len(d["tr_a"]); ntest = len(d["te_a"])
    tr_noisy, te_noisy = d["tr_noisy"], d["te_noisy"]
    z_oof_all, z_syn_all, z_te_all = [], [], []
    for mname in ACTIVE:
        Z = np.load(f"{OUT}/scores/{mname}_{cat}.npz")
        P = norm_params(Z["oof"][:, SCORE_KEY], tr_noisy)
        z_oof = apply_norm(Z["oof"][:, SCORE_KEY], tr_noisy, P)
        z_syn = apply_norm(Z["syn"][:, :, SCORE_KEY], tr_noisy[:, None], P)
        z_te = apply_norm(Z["test"][:, SCORE_KEY], te_noisy, P)
        z_oof_all.append(z_oof); z_syn_all.append(z_syn); z_te_all.append(z_te)

    def evaluate(name, z_oof, z_syn):
        s = np.concatenate([z_oof, z_syn.reshape(-1)])
        y = np.concatenate([np.zeros(len(z_oof)), np.ones(z_syn.size)])
        wg = (z_syn.size * (1 - TARGET_DEF_FRAC)) / (TARGET_DEF_FRAC * len(z_oof))
        w = np.where(y == 1, 1.0, wg)
        f1, t = best_f1(s, y, w)
        rows.append(dict(category=cat, model=name, auroc=roc_auc_score(y, s), best_f1_weighted=f1))
        return s, y, w

    for k, mname in enumerate(ACTIVE):
        evaluate(mname, z_oof_all[k], z_syn_all[k])
    z_oof = np.mean(z_oof_all, 0); z_syn = np.mean(z_syn_all, 0); z_te = np.mean(z_te_all, 0)
    s, y, w = evaluate("ENSEMBLE", z_oof, z_syn)

    t_val = best_f1(s, y, w)[1]
    t_pct = float(np.percentile(z_oof, PCT))
    t_gmm, sep, wh = gmm_threshold(z_te)
    gmm_ok = (sep > 2.5) and (wh > 0.4)
    cands = [t_val, t_pct] + ([t_gmm] if gmm_ok else [])
    t_blend = float(np.median(cands))
    thr_info[cat] = dict(t_val=float(t_val), t_pct=t_pct, t_gmm=float(t_gmm), gmm_reliable=bool(gmm_ok),
                         gmm_sep=float(sep), t_blend=t_blend)
    pred_modes["val"][cat] = z_te >= t_val
    pred_modes["pct"][cat] = z_te >= t_pct
    pred_modes["gmm"][cat] = z_te >= (t_gmm if gmm_ok else t_pct)
    pred_modes["blend"][cat] = z_te >= t_blend
    ens_test[cat] = z_te

    # suspects = good training images that look anomalous (possible hidden defects / label noise)
    top = np.argsort(-z_oof)[:8]
    suspects[cat] = [(d["tr"]["id"][i], round(float(z_oof[i]), 2)) for i in top]

    fig, ax = plt.subplots(figsize=(8, 3))
    bins = np.linspace(min(z_oof.min(), z_syn.min(), z_te.min()), max(np.percentile(z_syn, 99), z_te.max()), 60)
    ax.hist(z_oof, bins, alpha=.5, label="held-out good (train CV)", density=True)
    ax.hist(z_syn.reshape(-1), bins, alpha=.4, label="synthetic defects", density=True)
    ax.hist(z_te, bins, alpha=.4, label="TEST", density=True)
    for nme, t, c in [("val", t_val, "g"), ("pct", t_pct, "orange"), ("gmm", t_gmm, "purple"), ("blend", t_blend, "k")]:
        ax.axvline(t, color=c, ls="--", label=f"T_{nme}={t:.1f}")
    ax.set_title(f"{cat}: ensemble z-scores"); ax.legend(fontsize=6); plt.tight_layout()
    plt.savefig(f"{OUT}/figs/hist_{cat}.png", dpi=80); plt.show()

res = pd.DataFrame(rows)
print(res.pivot(index="category", columns="model", values="auroc").round(4))
print(res.pivot(index="category", columns="model", values="best_f1_weighted").round(4))
res.to_csv(f"{OUT}/pseudo_validation_results.csv", index=False)
json.dump(thr_info, open(f"{OUT}/thresholds.json", "w"), indent=2)
print(json.dumps(thr_info, indent=1))
print("\nGood TRAIN images with the highest scores (check them by eye; remove obvious defects & re-run):")
for c, v in suspects.items():
    print(" ", c, v)

In [ ]:
def write_submission(mode, fname):
    parts = []
    for cat in CATS:
        te = DATA[cat]["te"]
        parts.append(pd.DataFrame({"id": te["id"], "label": np.where(pred_modes[mode][cat], "defective", "good")}))
    sub = pd.concat(parts)
    if os.path.exists(sample_path):
        sub = pd.read_csv(sample_path)[["id"]].merge(sub, on="id", how="left")
    else:
        sub = test[["id"]].merge(sub, on="id", how="left")
    assert sub["label"].notna().all() and len(sub) == len(test) and set(sub.label) <= {"good", "defective"}
    sub.to_csv(f"{OUT}/{fname}", index=False)
    frac = {c: round(float(pred_modes[mode][c].mean()), 2) for c in CATS}
    print(f"{fname:26s} defective share per category: {frac} | overall {np.mean(sub.label == 'defective'):.2f}")

for mode in ["val", "pct", "gmm", "blend"]:
    write_submission(mode, f"submission_{mode}.csv")
write_submission("blend", "submission.csv")      # <- the file Kaggle picks up as the default output

pd.concat([pd.DataFrame({"id": DATA[c]["te"]["id"], "category": c, "z": ens_test[c]}) for c in CATS]).to_csv(
    f"{OUT}/test_ensemble_scores.csv", index=False)
print("\nSubmit submission.csv first. Then compare submission_val / _pct / _gmm on the public leaderboard "
      "(10 per day), but pick your 2 final submissions by reasoning + consistency, not by tiny public gains.")